Cek apakah Nvidia CUDA tersedia

In [1]:
!nvidia-smi

Fri May  1 13:39:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Mounting google drive dengan notebook (Colab only)

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
import yaml

YAML_FOLDER = '/content/drive/MyDrive/Training Data Colab/'
YAML_FILENAME = YAML_FOLDER + 'simple_hi_peace1.yml'

with open(YAML_FILENAME, "r") as f:
    yaml_data = yaml.load(f, Loader=yaml.FullLoader)

model_name = yaml_data["name"]
model_path = yaml_data["path"]
training_data = yaml_data["training_data"]
hidden_layers = yaml_data["hidden_layer"]
labels = yaml_data["labels"]

print(model_name)
print(model_path)
print(training_data)
print(hidden_layers)
print(labels)

simple_hi_peace1
simple_hi_peace1.keras
simple_hi_peace1.trdata.csv
[{'relu': 64}, {'dropout': 0.2}, {'relu': 64}, {'dropout': 0.2}, {'relu': 32}]
['Hi', 'Peace']


In [4]:
import pandas as pd

data = pd.read_csv(YAML_FOLDER + training_data, header=None)

X = data.iloc[:, 1:].values
Y = data.iloc[:, 0].values

In [6]:
print(X)
print(Y)

print(X.shape)
print(Y.shape)

[[ 4.58249465e-01  1.00000000e+00  7.41467812e-07 ...  2.99980732e-01
   5.44462349e-01 -1.21248557e-01]
 [ 4.52631847e-01  1.00000000e+00  7.25615441e-07 ...  3.00327629e-01
   5.48696478e-01 -1.29876312e-01]
 [ 4.53749922e-01  1.00000000e+00  7.32472127e-07 ...  3.02964769e-01
   5.51473581e-01 -1.22969420e-01]
 ...
 [ 9.24137757e-01  6.44583503e-01  8.30817439e-07 ...  1.00000000e+00
   5.50072818e-01 -9.79867765e-02]
 [ 9.28920053e-01  6.42592660e-01  7.79527771e-07 ...  1.00000000e+00
   5.76455474e-01 -9.25577128e-02]
 [ 9.89453431e-01  8.44866355e-01  3.67602668e-07 ...  9.82009483e-01
   6.82561015e-01 -1.73643995e-02]]
[0 0 0 ... 1 1 1]
(4000, 63)
(4000,)


In [8]:
import tensorflow as tf
from tensorflow.keras import models, layers

model = models.Sequential()
for layer_spec in hidden_layers:
    layer_type, params = next(iter(layer_spec.items()))
    if layer_type == "dropout":
        model.add(layers.Dropout(params))
    else:
        kwargs = {"activation": layer_type}
        if not model.layers:
            kwargs["input_shape"] = (X.shape[1],)
        model.add(layers.Dense(params, **kwargs))

model.add(layers.Dense(len(labels), activation="softmax"))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [9]:
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [10]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │         4,096 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 2)              │            66 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,402 (40.63 KB)

 Trainable params: 10,402 (40.63 KB)

 Non-trainable params: 0 (0.00 B)

In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y,
    test_size= 0.2,
    shuffle=True,
    stratify=Y
)

In [12]:
from tensorflow.keras.callbacks import EarlyStopping
es = EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)

In [13]:
model.fit(X_train, Y_train, epochs=100, validation_data=(X_test, Y_test), callbacks=[es])

Epoch 1/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.6112 - loss: 0.6482 - val_accuracy: 0.7175 - val_loss: 0.5422
Epoch 2/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7903 - loss: 0.4510 - val_accuracy: 0.9237 - val_loss: 0.2379
Epoch 3/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9169 - loss: 0.2259 - val_accuracy: 0.9725 - val_loss: 0.0992
Epoch 4/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9500 - loss: 0.1393 - val_accuracy: 0.9750 - val_loss: 0.0861
Epoch 5/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9656 - loss: 0.1026 - val_accuracy: 0.9900 - val_loss: 0.0606
Epoch 6/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9722 - loss: 0.0827 - val_accuracy: 0.9887 - val_loss: 0.0433
Epoch 7/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9722 - loss: 0.0757 - val_accuracy: 0.9825 - val_loss: 0.0515
Epoch 8/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9859 - loss: 0.0524 - val_accu

In [14]:
model.save(model_path)